# Consumer Complaints ML Features and EDA

This notebook is fully self-contained and runs directly in Databricks.

It profiles the Gold warehouse layer, explores the modelling target, builds a reusable feature table, and validates the saved output.

## Step 1: Imports, constants, and Spark session

In [ ]:
import logging  # Standard logging keeps the notebook output readable during reruns.

from pyspark.sql import SparkSession  # Core Spark entry point for DataFrame work.
from pyspark.sql import functions as F  # Spark SQL expressions used throughout the notebook.
from pyspark.sql.window import Window  # Window logic is used for percentage calculations.

spark = SparkSession.builder.getOrCreate()  # Reuse the active Databricks Spark session.

CATALOG = "fintech_lakehouse_dev"  # Development catalog for this lakehouse project.
GOLD_SCHEMA = "gold"  # Gold schema contains cleaned warehouse-ready entities.

FACT_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.fact_consumer_complaints"  # Main complaint fact table.
DIM_DATE_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"  # Date dimension for complaint intake dates.
DIM_PRODUCT_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_product"  # Product and sub-product dimension.
DIM_COMPANY_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_company"  # Company dimension.
DIM_STATE_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_state"  # State dimension.

TARGET_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.consumer_complaints_timely_response_features"  # Managed feature table used by training.

MISSING_MEMBER_LABEL = "NOT_PROVIDED"  # Keep missing categoricals explicit instead of null.
TRAIN_SPLIT_LABEL = "TRAIN"  # Training split label.
VALIDATION_SPLIT_LABEL = "VALIDATION"  # Validation split label.
TEST_SPLIT_LABEL = "TEST"  # Test split label.

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")  # Notebook-friendly logging format.
LOGGER = logging.getLogger("consumer_complaints_ml_features_notebook")  # Dedicated logger for this notebook.

print("Feature target table:", TARGET_TABLE)


## Step 2: Validate Gold source tables

In [ ]:
required_tables = [FACT_TABLE, DIM_DATE_TABLE, DIM_PRODUCT_TABLE, DIM_COMPANY_TABLE, DIM_STATE_TABLE]  # All warehouse inputs required by feature engineering.

for table_name in required_tables:
    if not spark.catalog.tableExists(table_name):  # Fail early if the Gold layer is incomplete.
        raise RuntimeError(f"Required Gold table does not exist: {table_name}")

fact_row_count = spark.table(FACT_TABLE).count()  # Capture the source row count for later validation.
if fact_row_count == 0:
    raise RuntimeError(f"Gold fact table is empty: {FACT_TABLE}")

LOGGER.info("Validated Gold sources. Fact table %s contains %s rows.", FACT_TABLE, f"{fact_row_count:,}")
print("Gold fact rows:", f"{fact_row_count:,}")


## Step 3: Build the joined Gold modelling base

In [ ]:
fact_dataframe = spark.table(FACT_TABLE).alias("f")  # Start from the Gold fact table at complaint grain.

dim_date_dataframe = (
    spark.table(DIM_DATE_TABLE)
    .select(
        F.col("date_key").alias("received_date_key"),  # Rename the join key to match the fact table.
        F.col("full_date").alias("received_full_date"),  # Human-readable complaint intake date.
        F.col("calendar_year").alias("received_year"),  # Year feature for modelling seasonality.
        F.col("calendar_quarter").alias("received_quarter"),  # Quarter feature for coarse seasonality.
        F.col("calendar_month").alias("received_month"),  # Month-of-year feature.
        F.col("calendar_month_name").alias("received_month_name"),  # Readable month label for EDA and categoricals.
        F.col("calendar_day").alias("received_day"),  # Day-of-month feature.
        F.col("day_of_week").alias("received_day_of_week"),  # Weekday index feature.
        F.col("day_name").alias("received_day_name"),  # Readable weekday label.
        F.col("is_weekend").alias("received_is_weekend"),  # Binary weekend indicator.
    )
    .alias("dd")
)

dim_product_dataframe = spark.table(DIM_PRODUCT_TABLE).alias("dp")  # Product dimension adds product labels.
dim_company_dataframe = spark.table(DIM_COMPANY_TABLE).alias("dc")  # Company dimension adds company names.
dim_state_dataframe = spark.table(DIM_STATE_TABLE).alias("ds")  # State dimension adds standardised state codes.

feature_base = (
    fact_dataframe
    .join(dim_date_dataframe, on="received_date_key", how="left")  # Attach the complaint intake calendar attributes.
    .join(dim_product_dataframe, on="product_key", how="left")  # Attach product and sub-product labels.
    .join(dim_company_dataframe, on="company_key", how="left")  # Attach company labels.
    .join(dim_state_dataframe, on="state_key", how="left")  # Attach state labels.
)

display(feature_base.select(
    "complaint_id",  # Business key used later for train/test determinism.
    "timely_response",  # Raw target column from the warehouse model.
    "product",  # High-level complaint product category.
    "sub_product",  # More granular complaint subtype.
    "company",  # Company receiving the complaint.
    "state",  # Complaint geography.
    "issue",  # Complaint issue bucket.
    "submitted_via",  # Intake channel.
    "received_full_date"  # Intake date for temporal features.
).limit(20))


## Step 4: Explore the modelling target

In [ ]:
target_profile = (
    feature_base
    .groupBy("timely_response")  # Profile each target class before modelling.
    .count()  # Count complaints in each target bucket.
    .withColumn(
        "pct_of_rows",
        F.round(F.col("count") / F.sum("count").over(Window.partitionBy()) * 100, 4),  # Convert counts into percentages to expose imbalance.
    )
)

display(target_profile.orderBy("timely_response"))


## Step 5: Review key categorical distributions

In [ ]:
display(feature_base.groupBy("product").count().orderBy(F.desc("count")).limit(20))  # Largest complaint product groups can dominate model learning.
display(feature_base.groupBy("state").count().orderBy(F.desc("count")).limit(20))  # State concentration can create geographic bias or signal.
display(feature_base.groupBy("submitted_via").count().orderBy(F.desc("count")))  # Intake channel may correlate with response timeliness.


## Step 6: Review product-level untimely response rates

In [ ]:
product_target_profile = (
    feature_base
    .filter(F.col("timely_response").isin("YES", "NO"))  # Restrict to clean binary-labelled complaints.
    .groupBy("product")  # Compare target behaviour across complaint categories.
    .agg(
        F.count("*").alias("row_count"),  # Overall complaint count per product.
        F.sum(F.when(F.col("timely_response") == "NO", 1).otherwise(0)).alias("untimely_count"),  # Number of untimely complaints per product.
    )
    .withColumn("untimely_rate_pct", F.round(F.col("untimely_count") / F.col("row_count") * 100, 4))  # Convert to a rate that is easier to compare.
    .orderBy(F.desc("untimely_rate_pct"), F.desc("row_count"))  # Prioritise high-risk products with meaningful volume.
)

display(product_target_profile.limit(25))


## Step 7: Build the managed ML feature DataFrame

In [ ]:
feature_dataframe = (
    feature_base
    .filter(F.col("timely_response").isin("YES", "NO"))  # Keep only rows that can contribute to binary classification.
    .withColumn("label_untimely_response", F.when(F.col("timely_response") == "NO", F.lit(1)).otherwise(F.lit(0)))  # Derive the numeric label expected by Spark ML.
    .withColumn("complaint_received_date", F.col("received_full_date"))  # Preserve the readable intake date as a feature field.
    .withColumn("zip_code_prefix3", F.when(F.col("zip_code").isNull(), F.lit(MISSING_MEMBER_LABEL)).otherwise(F.substring(F.col("zip_code"), 1, 3)))  # Reduce postal code granularity while keeping leading digits.
    .withColumn("product", F.coalesce(F.col("product"), F.lit(MISSING_MEMBER_LABEL)))  # Replace missing product labels with an explicit placeholder.
    .withColumn("sub_product", F.coalesce(F.col("sub_product"), F.lit(MISSING_MEMBER_LABEL)))  # Preserve sparsity explicitly instead of losing rows.
    .withColumn("company", F.coalesce(F.col("company"), F.lit(MISSING_MEMBER_LABEL)))  # Keep company available as a categorical model feature.
    .withColumn("state", F.coalesce(F.col("state"), F.lit(MISSING_MEMBER_LABEL)))  # Missing states become a stable category.
    .withColumn("issue", F.coalesce(F.col("issue"), F.lit(MISSING_MEMBER_LABEL)))  # Issue text bucket is often predictive.
    .withColumn("sub_issue", F.coalesce(F.col("sub_issue"), F.lit(MISSING_MEMBER_LABEL)))  # Keep complaint sub-issue detail where present.
    .withColumn("submitted_via", F.coalesce(F.col("submitted_via"), F.lit(MISSING_MEMBER_LABEL)))  # Preserve intake channel even when missing.
    .withColumn("received_month_name", F.coalesce(F.col("received_month_name"), F.lit(MISSING_MEMBER_LABEL)))  # Readable month label for model encoding.
    .withColumn("received_day_name", F.coalesce(F.col("received_day_name"), F.lit(MISSING_MEMBER_LABEL)))  # Readable weekday label for model encoding.
    .withColumn("received_is_weekend", F.coalesce(F.col("received_is_weekend").cast("int"), F.lit(0)))  # Cast booleans into numeric model-friendly form.
    .withColumn("has_consumer_narrative", F.col("has_consumer_narrative").cast("int"))  # Convert booleans into numeric indicators.
    .withColumn("has_tags", F.col("has_tags").cast("int"))  # Convert booleans into numeric indicators.
    .withColumn("complaint_received_year", F.coalesce(F.col("received_year"), F.lit(0)))  # Fill rare missing calendar values safely.
    .withColumn("complaint_received_quarter", F.coalesce(F.col("received_quarter"), F.lit(0)))  # Quarter feature for seasonality.
    .withColumn("complaint_received_month", F.coalesce(F.col("received_month"), F.lit(0)))  # Month feature for seasonality.
    .withColumn("complaint_received_day", F.coalesce(F.col("received_day"), F.lit(0)))  # Day-of-month feature.
    .withColumn("complaint_received_day_of_week", F.coalesce(F.col("received_day_of_week"), F.lit(0)))  # Day-of-week numeric feature.
    .withColumn("split_bucket", F.pmod(F.xxhash64(F.col("complaint_id")), F.lit(100)))  # Deterministically bucket each complaint for repeatable splits.
    .withColumn(
        "dataset_split",
        F.when(F.col("split_bucket") < 70, F.lit(TRAIN_SPLIT_LABEL))
         .when(F.col("split_bucket") < 85, F.lit(VALIDATION_SPLIT_LABEL))
         .otherwise(F.lit(TEST_SPLIT_LABEL)),  # Create a stable 70/15/15 style split without random drift between reruns.
    )
    .withColumn("_feature_processed_at", F.current_timestamp())  # Record when the feature table was generated.
    .select(
        "complaint_id",
        "label_untimely_response",
        "timely_response",
        "complaint_received_date",
        "complaint_received_year",
        "complaint_received_quarter",
        "complaint_received_month",
        "complaint_received_day",
        "complaint_received_day_of_week",
        "received_month_name",
        "received_day_name",
        "received_is_weekend",
        "product",
        "sub_product",
        "company",
        "state",
        "issue",
        "sub_issue",
        "submitted_via",
        "zip_code_prefix3",
        "has_consumer_narrative",
        "has_tags",
        "dataset_split",
        "split_bucket",
        "_ingestion_date",
        "_ingested_at",
        "_source_zip_path",
        "_source_csv_name",
        "_silver_processed_at",
        "_gold_processed_at",
        "_feature_processed_at",
    )
)

LOGGER.info("Built ML feature DataFrame for timely-response prediction.")
display(feature_dataframe.limit(20))


## Step 8: Validate feature completeness and split balance

In [ ]:
feature_null_summary = feature_dataframe.select(
    F.count("*").alias("row_count"),  # Final feature row count before writing.
    F.count_if(F.col("product").isNull()).alias("null_product_count"),  # Product should no longer be null after coalesce.
    F.count_if(F.col("sub_product").isNull()).alias("null_sub_product_count"),  # Tracks any unexpected missing subtype values.
    F.count_if(F.col("company").isNull()).alias("null_company_count"),  # Company should no longer be null after coalesce.
    F.count_if(F.col("state").isNull()).alias("null_state_count"),  # State should no longer be null after coalesce.
    F.count_if(F.col("issue").isNull()).alias("null_issue_count"),  # Issue should no longer be null after coalesce.
    F.count_if(F.col("submitted_via").isNull()).alias("null_submitted_via_count"),  # Intake channel should no longer be null after coalesce.
)

display(feature_null_summary)
display(feature_dataframe.groupBy("dataset_split").count().orderBy("dataset_split"))  # Validate split volumes.
display(feature_dataframe.groupBy("dataset_split", "label_untimely_response").count().orderBy("dataset_split", "label_untimely_response"))  # Validate class balance within each split.


## Step 9: Write and validate the managed feature table

In [ ]:
(
    feature_dataframe.write
    .format("delta")  # Persist the features as a managed Delta table.
    .mode("overwrite")  # Overwrite keeps the feature build idempotent.
    .option("overwriteSchema", "true")  # Allow safe schema refresh if the notebook evolves.
    .saveAsTable(TARGET_TABLE)
)

saved_feature_dataframe = spark.table(TARGET_TABLE)  # Re-open the saved table for post-write validation.
saved_feature_row_count = saved_feature_dataframe.count()  # Persisted row count must be positive and not exceed Gold fact volume.

if saved_feature_row_count == 0:
    raise RuntimeError("Feature validation failed: target row count is zero.")

if saved_feature_row_count > fact_row_count:
    raise RuntimeError("Feature validation failed: feature row count exceeds the Gold fact row count.")

null_complaint_ids = saved_feature_dataframe.filter(F.col("complaint_id").isNull()).count()  # Complaint key must remain populated.
if null_complaint_ids > 0:
    raise RuntimeError(f"Feature validation failed: {null_complaint_ids} null complaint IDs found.")

duplicate_complaint_ids = (
    saved_feature_dataframe.groupBy("complaint_id")
    .count()
    .filter(F.col("count") > 1)  # Feature table must remain one row per complaint.
    .count()
)
if duplicate_complaint_ids > 0:
    raise RuntimeError(f"Feature validation failed: {duplicate_complaint_ids} duplicate complaint IDs found.")

invalid_labels = saved_feature_dataframe.filter((~F.col("label_untimely_response").isin(0, 1)) | F.col("label_untimely_response").isNull()).count()  # Label must stay binary and non-null.
if invalid_labels > 0:
    raise RuntimeError(f"Feature validation failed: {invalid_labels} invalid label values found.")

split_counts = {row["dataset_split"]: row["row_count"] for row in saved_feature_dataframe.groupBy("dataset_split").count().withColumnRenamed("count", "row_count").collect()}  # Convert split counts into a small Python dict for checks.

for required_split in [TRAIN_SPLIT_LABEL, VALIDATION_SPLIT_LABEL, TEST_SPLIT_LABEL]:
    if split_counts.get(required_split, 0) == 0:
        raise RuntimeError(f"Feature validation failed: required split {required_split} is empty.")

LOGGER.info(
    "Feature validation successful: %s rows written. Split sizes: train=%s, validation=%s, test=%s.",
    f"{saved_feature_row_count:,}",
    f"{split_counts.get(TRAIN_SPLIT_LABEL, 0):,}",
    f"{split_counts.get(VALIDATION_SPLIT_LABEL, 0):,}",
    f"{split_counts.get(TEST_SPLIT_LABEL, 0):,}",
)

print("Feature table written to:", TARGET_TABLE)
display(saved_feature_dataframe.limit(20))
